<a href="https://colab.research.google.com/github/SmritiD2004/RAG--IBMSkillsbuild/blob/main/Dynamic_RAG_Web_Crawlingipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu google-generativeai beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 35.9 MB/s eta 0:00:00


In [2]:
import faiss
import numpy as np
import requests
import re
import google.generativeai as genai

from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini initialized successfully!")

Gemini initialized successfully!


In [4]:
user_query = input("Enter your query: ")

response = model.generate_content(user_query)

baseline_response = response.text

print("\n===== Baseline Response (Without RAG) =====\n")
print(baseline_response)

Enter your query: Tell me more about AI Alliance

===== Baseline Response (Without RAG) =====

The **AI Alliance** is a broad, international consortium of leading organizations across the tech industry, academia, research, and government, launched in **December 2023**. It was co-founded by **IBM and Meta**, and its primary mission is to foster an **open, safe, and responsible approach to AI development and deployment.**

Here's a breakdown of what the AI Alliance is all about:

1.  **Core Mission and Goals:**
    *   **Open Science and Innovation:** The central tenet is to promote an open AI ecosystem, emphasizing the sharing of open-source AI models, tools, data, and research. This is seen as a way to democratize access to AI, accelerate innovation, and prevent a few companies from monopolizing the technology.
    *   **Safety and Responsibility:** A strong focus on ensuring AI is developed and deployed responsibly, addressing potential risks like bias, privacy, security, and the spre

In [5]:
url = "https://thealliance.ai/"

try:
    response = requests.get(url, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove unwanted HTML tags
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    page_text = soup.get_text(separator="\n")

    page_text = re.sub(r"\n\s*\n+", "\n\n", page_text)

    with open("thealliance_ai_content.txt", "w", encoding="utf-8") as f:
        f.write(page_text)

    print("Website crawled successfully!")
    print(f"Characters extracted: {len(page_text)}")

except Exception as e:
    print("Web crawling failed:", e)

Website crawled successfully!
Characters extracted: 6058


In [6]:
def split_into_chunks(text, chunk_size=300, overlap=50):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks


chunks = split_into_chunks(page_text)

print(f"Created {len(chunks)} chunks.")

Created 25 chunks.


In [7]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embed_model.encode(
    chunks,
    convert_to_numpy=True
).astype(np.float32)

print("Embeddings Shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings Shape: (25, 384)


In [8]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

chunk_store = chunks

print(f"Indexed {index.ntotal} document chunks.")

Indexed 25 document chunks.


In [9]:
query_embedding = embed_model.encode(
    [user_query],
    convert_to_numpy=True
).astype(np.float32)

k = 3

distances, indices = index.search(query_embedding, k)

retrieved_chunks = [chunk_store[i] for i in indices[0]]

print("Top Retrieved Chunks:\n")

for i, chunk in enumerate(retrieved_chunks, start=1):
    print("=" * 60)
    print(f"Chunk {i}")
    print("=" * 60)
    print(chunk)
    print()

Top Retrieved Chunks:

Chunk 1


AI Alliance

About

About | AI Alliance | Leadership

Projects

Events

Join

Blog

AI Alliance

A global network advancing open, safe, and responsible AI through open innovation, collaboration, and advocacy

Home

AI Alliance

To accelerate open innovation in AI, advancing core capabilities and e

Chunk 2
nnovation in AI, advancing core capabilities and establishing trusted frameworks for safety and security, to ensure responsible and enduring benefits for society at large. 

The AI Alliance is a non-profit 501(c)(3) organization dedicated to building and advancing open source AI agents, data, models

Chunk 3
s been true catalysts for progress. The launch of the AI Alliance marks a visionary milestone, uniting industry giants, academia, and innovators with a shared responsibility to shape the future and advance open innovation, ensuring that the transformative power of AI is harnessed responsibly and eth



In [10]:
retrieved_context = "\n\n".join(retrieved_chunks)

prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the provided context.

If the answer is not available in the context, reply:

"I don't have enough information in the provided context."

Context:
{retrieved_context}

Question:
{user_query}
"""

In [11]:
response = model.generate_content(prompt)

rag_response = response.text

print("\n===== RAG Response =====\n")

print(rag_response)


===== RAG Response =====

The AI Alliance is a global network that advances open, safe, and responsible AI through open innovation, collaboration, and advocacy. It aims to accelerate open innovation in AI, advance core capabilities, and establish trusted frameworks for safety and security to ensure responsible and enduring benefits for society.

It is a non-profit 501(c)(3) organization dedicated to building and advancing open source AI agents, data, and models. The AI Alliance unites industry giants, academia, and innovators with a shared responsibility to shape the future and advance open innovation, ensuring the transformative power of AI is harnessed responsibly and ethically.


In [14]:
from IPython.display import Markdown, display

display(Markdown(f"""
# User Query

**{user_query}**

---

# Baseline Response (Without RAG)

{baseline_response}

---

# Retrieved Context

{retrieved_context}

---

# RAG Response

{rag_response}
"""))


# User Query

**Tell me more about AI Alliance**

---

# Baseline Response (Without RAG)

The **AI Alliance** is a broad, international consortium of leading organizations across the tech industry, academia, research, and government, launched in **December 2023**. It was co-founded by **IBM and Meta**, and its primary mission is to foster an **open, safe, and responsible approach to AI development and deployment.**

Here's a breakdown of what the AI Alliance is all about:

1.  **Core Mission and Goals:**
    *   **Open Science and Innovation:** The central tenet is to promote an open AI ecosystem, emphasizing the sharing of open-source AI models, tools, data, and research. This is seen as a way to democratize access to AI, accelerate innovation, and prevent a few companies from monopolizing the technology.
    *   **Safety and Responsibility:** A strong focus on ensuring AI is developed and deployed responsibly, addressing potential risks like bias, privacy, security, and the spread of misinformation.
    *   **Collaboration and Community Building:** To bring together diverse stakeholders globally to tackle complex AI challenges collectively.
    *   **Standards and Benchmarks:** Working towards developing globally accepted standards, benchmarks, and evaluation tools for AI systems, particularly concerning safety, trustworthiness, and interoperability.
    *   **Talent Development:** Fostering a global talent pool in AI through education, training, and research initiatives.
    *   **Addressing Societal Challenges:** Leveraging AI to solve real-world problems while also considering its ethical, societal, and economic impacts.

2.  **Founding Members and Scope:**
    *   The Alliance boasts an impressive and diverse list of over 50 founding members (and growing).
    *   **Key Tech Companies:** IBM, Meta, AMD, Intel, Oracle, Sony, Stability AI, Hugging Face, Cerebras Systems, ServiceNow, Databricks, Red Hat, and more.
    *   **Academic Institutions:** Cornell University, Imperial College London, Princeton University, Stanford University, University of California, Berkeley, University of Tokyo, Yale University, and others.
    *   **Research Organizations:** Allen Institute for AI (AI2), CERN, NASA, National Science Foundation (NSF), MLCommons, and others.
    *   This broad representation is critical to its aim of building a truly inclusive and globally impactful initiative.

3.  **Key Areas of Focus and Activities:**
    *   **Developing and Sharing Open-Source Technologies:** This includes foundational models, libraries, and tools that can be used by anyone. Meta's open-source Llama 2 large language model is a prime example of the kind of contribution aligned with the Alliance's philosophy.
    *   **Research and Development:** Supporting and coordinating cutting-edge research in areas like AI safety, explainability, privacy-preserving AI, and new AI architectures.
    *   **Policy Advocacy:** Engaging with governments and policymakers worldwide to advocate for regulations and policies that support open innovation and responsible AI development.
    *   **Community Engagement:** Hosting workshops, conferences, hackathons, and educational programs to foster collaboration and knowledge sharing.
    *   **Governance and Trust:** Creating frameworks for AI governance and developing mechanisms to ensure transparency and accountability.

4.  **Why it Matters (Significance):**
    *   **Counterbalance to Closed AI:** The AI Alliance represents a significant counter-movement to the trend of a few large companies developing powerful AI models behind closed doors (e.g., OpenAI's GPT models or Google's Gemini). It aims to prevent a "walled garden" approach to AI.
    *   **Democratization of AI:** By promoting open-source tools, it makes advanced AI accessible to a wider range of researchers, startups, and smaller organizations, fostering broader innovation.
    *   **Enhanced Safety and Ethics:** More eyes on the code and broader collaboration can lead to more robust testing, identification of biases, and development of safer AI systems.
    *   **Global Influence:** Its international membership gives it potential influence in shaping global AI norms, standards, and regulatory frameworks.

In essence, the AI Alliance is a major collaborative effort to steer the future of artificial intelligence towards a more open, transparent, and globally responsible path, ensuring that the benefits of AI are widely shared and its risks are collectively managed.

---

# Retrieved Context



AI Alliance

About

About | AI Alliance | Leadership

Projects

Events

Join

Blog

AI Alliance

A global network advancing open, safe, and responsible AI through open innovation, collaboration, and advocacy

Home

AI Alliance

To accelerate open innovation in AI, advancing core capabilities and e

nnovation in AI, advancing core capabilities and establishing trusted frameworks for safety and security, to ensure responsible and enduring benefits for society at large. 

The AI Alliance is a non-profit 501(c)(3) organization dedicated to building and advancing open source AI agents, data, models

s been true catalysts for progress. The launch of the AI Alliance marks a visionary milestone, uniting industry giants, academia, and innovators with a shared responsibility to shape the future and advance open innovation, ensuring that the transformative power of AI is harnessed responsibly and eth

---

# RAG Response

The AI Alliance is a global network that advances open, safe, and responsible AI through open innovation, collaboration, and advocacy. It aims to accelerate open innovation in AI, advance core capabilities, and establish trusted frameworks for safety and security to ensure responsible and enduring benefits for society.

It is a non-profit 501(c)(3) organization dedicated to building and advancing open source AI agents, data, and models. The AI Alliance unites industry giants, academia, and innovators with a shared responsibility to shape the future and advance open innovation, ensuring the transformative power of AI is harnessed responsibly and ethically.
